# 혜진: 암종 분류 v2 — 표기 정규화와 분포 변화에 견디는 모델
이 노트북 하나에서 EDA, 파생변수 생성/선별, CV 모델 선택, 검증, 전체 재학습, 피클·제출·보고서 저장을 실행합니다. 다른 프로젝트 Python 파일은 필요하지 않습니다.

이전 Public 점수 **0.2765425197**, 로컬 홀드아웃 macro F1 **0.4237006861**을 기준으로 개선을 시도합니다. 로컬 결과와 대회 결과를 구분합니다. 기존 홀드아웃은 이미 관측되어 새로운 독립 평가가 아닙니다. 이번 후보 비교는 개발 그룹 CV와 합성 배경 변이 스트레스 점수로만 진행하고 기존 홀드아웃은 보고용으로 재평가합니다.

필요 라이브러리: numpy, pandas, scipy, scikit-learn, lightgbm, matplotlib, cloudpickle, threadpoolctl, IPython. macOS LightGBM 로딩 시 libomp가 필요할 수 있습니다. 데이터 CSV는 공용 data/를 사용합니다. 개인 결과는 models/혜진/artifacts/, 제출본은 저장소 submission/에 저장합니다. 원래 작업 폴더에서도 실행 가능합니다.

공식 평가: [DACON 규칙](https://dacon.io/competitions/official/236355/overview/rules), Macro F1, Public은 테스트 30%·Private은 100%. 제공 학습 CSV만으로 사전·빈도·모델을 학습합니다. 외부 데이터, 테스트 기반 학습/가중치 적합, 의사 라벨은 사용하지 않습니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from collections import Counter
from IPython.display import display,FileLink
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from sklearn.base import clone
from threadpoolctl import threadpool_limits
from lightgbm import LGBMClassifier
threadpool_limits(2)
SEED=42
# Find a common data/ from either the repo root or models/혜진/.
search_roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in search_roots if all((p/'data'/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])),None)
if REPO is None: raise FileNotFoundError('공용 data/에 train.csv, test.csv, sample_submission.csv가 필요합니다.')
WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v2';OUT.mkdir(parents=True,exist_ok=True)
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
SUBMIT=REPO/'submission';SUBMIT.mkdir(exist_ok=True)
train=pd.read_csv(REPO/'data/train.csv');test=pd.read_csv(REPO/'data/test.csv');sample=pd.read_csv(REPO/'data/sample_submission.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert list(test)==['ID']+genes
assert sample.ID.equals(test.ID) and train.ID.is_unique and test.ID.is_unique
X=train[genes];y=train.SUBCLASS;labels=np.sort(y.unique());yi=np.searchsorted(labels,y)
groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42).split(X,y,groups))
folds=list(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=43).split(X.iloc[dev],y.iloc[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('Repository:',REPO,'Output:',OUT,'Shape:',train.shape,test.shape,'Development/holdout:',len(dev),len(hold))

Repository: /Users/mac/project/cancer_project Output: /Users/mac/project/cancer_project/artifacts/v2 Shape: (6201, 4386) (2546, 4385) Development/holdout: 4960 1241


## 1. 주석 표기·분포 EDA
WT는 변이 없음 표기, 결측은 미관측으로 처리합니다. 변이 유전자 수는 임상적 TMB가 아닙니다. `X`와 `*`는 이 데이터의 아미노산 치환 끝 위치에 있을 때 같은 stop 범주로 처리합니다. 다중 토큰은 유전자별 유형 존재 여부로 합쳐 중복 주석 수에 대한 의존을 줄입니다. 동일 유전자/유형 플래그가 같으면 토큰이 여러 번 나와도 특성이 동일합니다. 실제 전사체 동일성은 제공 데이터만으로 확정할 수 없습니다.

In [2]:
def canonical_token(token):
 t=token.strip().removeprefix('p.')
 t=re.sub(r'(?<=\d)X$', '*',t)
 t=re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
 return t

def variant_kind(token):
 t=canonical_token(token)
 if 'fs' in t.lower():return 'frameshift'
 if '*' in t:return 'stop'
 if 'del' in t.lower():return 'deletion'
 if 'ins' in t.lower():return 'insertion'
 m=re.fullmatch(r'([A-Z])\d+([A-Z])',t)
 if m:return 'synonymous' if m[1]==m[2] else 'substitution'
 return 'other'

shift=[]
for name,df in [('train',X),('test',test[genes])]:
 arr=df.to_numpy();mask=pd.notna(arr)&(arr!='WT');values=arr[mask];tokens=[t for v in values for t in str(v).split()]
 counts=Counter(variant_kind(t) for t in tokens)
 row={'dataset':name,'rows':len(df),'mutated_cells':len(values),'tokens':len(tokens),'multi_cells':sum(len(str(v).split())>1 for v in values),'median_mutated_genes':float(np.median(mask.sum(axis=1))),'missing_cells':int(pd.isna(arr).sum()),'X_stops':sum(bool(re.fullmatch(r'[A-Z]\d+X',t)) for t in tokens),'star_stops':sum('*' in t for t in tokens),**counts}
 shift.append(row)
shift=pd.DataFrame(shift).fillna(0);shift.to_csv(OUT/'annotation_shift.csv',index=False);display(shift)
class_counts=y.value_counts().rename('count');class_counts.to_csv(OUT/'class_counts.csv')
# Preserve gene-level statistics for the report, without using test statistics for fitting.
gene_stats=pd.DataFrame({'gene':genes,'train_rate':X.ne('WT').where(X.notna(),False).mean().values,'test_rate':test[genes].ne('WT').where(test[genes].notna(),False).mean().values})
gene_stats.to_csv(OUT/'gene_statistics.csv',index=False)
assert canonical_token('R213X')==canonical_token('R213*')
assert canonical_token('K16Rfs')==canonical_token('K16fs')
print('Canonicalization checks passed.')

,dataset,rows,mutated_cells,tokens,multi_cells,median_mutated_genes,missing_cells,X_stops,star_stops,synonymous,substitution,stop,frameshift,other,deletion,insertion
0,train,6201,218893,255164,22026,14.0,0,0,13454,66883,164740,13451,9914,173,3,0.0
1,test,2546,198930,337512,63178,28.0,237,14477,1986,88724,202193,16462,25907,31,3098,1097.0


Canonicalization checks passed.


## 2. 세 가지 표현 및 파생변수
- gene: 유전자별 변이 유무.
- functional: gene + 유전자별 7개 표기 유형 존재 여부. 토큰 개수·절대 위치에 덜 의존합니다.
- canonical: functional + 정규화된 정확한 변이 + 유전자별 위치. 추가 정보의 이득과 표기 변화의 위험을 비교합니다.

모든 후보에서 원시 토큰 수·복수 이벤트 수·평균 아미노산 위치를 제거합니다. 이는 데이터 출처/주석 방식 차이를 암종 신호로 학습하는 위험을 줄이기 위한 실험입니다. 범주 빈도 필터(min_df=3), TF-IDF, chi2 변수 선별은 해당 폴드의 학습 부분만으로 적합합니다. ID와 SUBCLASS는 입력에서 제외합니다.

In [3]:
def row_features(frame,mode='functional'):
 names=list(frame.columns);rows=[]
 for values in frame.itertuples(index=False,name=None):
  d={}
  for g,v in zip(names,values):
   if not isinstance(v,str) or v=='WT':continue
   d['gene:'+g]=1.
   if mode=='gene':continue
   for t in set(canonical_token(t) for t in v.split()):
    k=variant_kind(t);d['type:'+g+':'+k]=1.
    if mode=='canonical':
     d['variant:'+g+':'+t]=1.
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=1.
  rows.append(d)
 return rows

# Deterministic per-row parsing uses no y or population-fitted statistics.
rows={mode:row_features(X,mode) for mode in ['gene','functional','canonical']}
class FeatureMap:
 def __init__(self,mode='functional',tfidf=False,k=None):self.mode=mode;self.tfidf=tfidf;self.k=k
 def fit(self,records,target):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(records)
  self.keep=np.asarray((a!=0).sum(axis=0)).ravel()>=3
  a=a[:,self.keep];self.names=self.vec.get_feature_names_out()[self.keep]
  self.selector=None
  if self.k is not None:
   self.selector=SelectKBest(chi2,k=min(self.k,a.shape[1])).fit(a,target);a=self.selector.transform(a);self.names=self.names[self.selector.get_support()]
  self.norm=None
  if self.tfidf:self.norm=TfidfTransformer().fit(a)
  return self
 def transform(self,records):
  a=sparse.csr_matrix(self.vec.transform(records)[:,self.keep])
  if self.selector is not None:a=self.selector.transform(a)
  if self.norm is not None:a=self.norm.transform(a)
  return sparse.csr_matrix(a,dtype=np.float32)

# Class-independent synthetic passenger noise. All gene probabilities come from fold training rows.
# No test distribution is used to fit noise rates. This is a sensitivity test, not a test-score estimate.
def add_background(records,training_records,seed):
 rng=np.random.default_rng(seed);c=Counter(k for d in training_records for k in d if k.startswith('gene:'))
 keys=list(c);prob=np.array([c[k] for k in keys],dtype=float);prob/=prob.sum();out=[]
 for d in records:
  nd=d.copy();n=min(len(keys),int(rng.poisson(20)))
  for key in rng.choice(keys,size=n,replace=False,p=prob):
   nd[key]=1.;g=key.split(':',1)[1]
   if any(k.startswith('type:') for k in d):nd['type:'+g+':'+('synonymous' if rng.random()<.5 else 'substitution')]=1.
  out.append(nd)
 return out

CONFIGS={
 'gene_lr':dict(mode='gene',tfidf=False,k=None,algo='lr',C=1,balanced=True,augment=False),
 'functional_lr':dict(mode='functional',tfidf=False,k=None,algo='lr',C=1,balanced=True,augment=False),
 'functional_tfidf_lr':dict(mode='functional',tfidf=True,k=None,algo='lr',C=4,balanced=True,augment=False),
 'canonical_tfidf_lr':dict(mode='canonical',tfidf=True,k=None,algo='lr',C=4,balanced=True,augment=False),
 'functional_aug_lr':dict(mode='functional',tfidf=False,k=None,algo='lr',C=.5,balanced=True,augment=True),
 'canonical_lgb15':dict(mode='canonical',tfidf=False,k=2000,algo='lgb',leaves=15,augment=False),
 'functional_lgb15':dict(mode='functional',tfidf=False,k=2000,algo='lgb',leaves=15,augment=False),
 'functional_lgb31':dict(mode='functional',tfidf=False,k=2000,algo='lgb',leaves=31,augment=False),
 'functional_aug_lgb':dict(mode='functional',tfidf=False,k=2000,algo='lgb',leaves=15,augment=True),
 'functional_extra':dict(mode='functional',tfidf=False,k=None,algo='extra',augment=False),
}
def estimator(cfg):
 if cfg['algo']=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced' if cfg['balanced'] else None,max_iter=2000)
 if cfg['algo']=='lgb':return LGBMClassifier(n_estimators=350,learning_rate=.05,num_leaves=cfg['leaves'],max_depth=-1,min_child_samples=15,colsample_bytree=.8,reg_lambda=5,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=300,min_samples_leaf=2,max_features=.15,class_weight='balanced',n_jobs=2,random_state=SEED)

def fit_member(name,idx):
 cfg=CONFIGS[name];tr=[rows[cfg['mode']][i] for i in idx];ty=yi[idx]
 mapping=FeatureMap(cfg['mode'],cfg['tfidf'],cfg['k']).fit(tr,ty)
 if cfg['augment']:
  aug=add_background(tr,tr,SEED+17);train_records=tr+aug;target=np.concatenate([ty,ty])
 else:train_records=tr;target=ty
 est=estimator(cfg).fit(mapping.transform(train_records),target)
 return {'name':name,'map':mapping,'estimator':est,'mode':cfg['mode']}

def predict_member(member,records):return member['estimator'].predict_proba(member['map'].transform(records))

CONFIGS['canonical_tfidf_lr_c16']=dict(CONFIGS['canonical_tfidf_lr'],C=16)
CONFIGS['canonical_tfidf_lr_c64']=dict(CONFIGS['canonical_tfidf_lr'],C=64)

CONFIGS['canonical_tfidf_lr_c025']=dict(CONFIGS['canonical_tfidf_lr'],C=.25)
CONFIGS['canonical_tfidf_lr_c1']=dict(CONFIGS['canonical_tfidf_lr'],C=1)


## 3. 개발 그룹 CV 및 스트레스 검증
기존과 같은 4,960명 개발/1,241명 홀드아웃과 3-fold 그룹 분할을 사용합니다. 원본 변이 프로파일이 동일한 샘플을 양쪽에 나누지 않습니다. 개발 점수는 clean macro F1과 class-independent 배경 변이를 추가한 stressed macro F1의 평균입니다. 스트레스 실험은 테스트 성능을 보장하지 않으며, 노이즈 20개는 테스트 통계로 추정하지 않은 고정 설정입니다. 후보/선택 규칙을 아래 코드에 명시합니다.

In [4]:
records=[]
oof={n:np.zeros((len(dev),len(labels)),dtype=np.float32) for n in CONFIGS}
stress_oof={n:np.zeros_like(oof[n]) for n in CONFIGS}
for fold,(a,b) in enumerate(folds):
 for name,cfg in CONFIGS.items():
  started=time.time();member=fit_member(name,dev[a])
  va=[rows[cfg['mode']][i] for i in dev[b]];tr=[rows[cfg['mode']][i] for i in dev[a]]
  stressed=add_background(va,tr,SEED+fold+100)
  p=predict_member(member,va);sp=predict_member(member,stressed)
  oof[name][b]=p;stress_oof[name][b]=sp
  clean=f1_score(yi[dev[b]],p.argmax(axis=1),average='macro',zero_division=0)
  stress=f1_score(yi[dev[b]],sp.argmax(axis=1),average='macro',zero_division=0)
  records.append({'model':name,'fold':fold,'clean_macro_f1':clean,'stress_macro_f1':stress,'robust_score':(clean+stress)/2,'features':len(member['map'].names),'seconds':time.time()-started})
  pd.DataFrame(records).to_csv(OUT/'cv_progress.csv',index=False)
  print(name,'fold',fold,'clean',round(clean,4),'stress',round(stress,4),'seconds',round(time.time()-started,1),flush=True)
  del member
scores=pd.DataFrame(records);scores.to_csv(OUT/'cv_folds.csv',index=False)
ranking=scores.groupby('model')[['clean_macro_f1','stress_macro_f1','robust_score']].mean().sort_values('robust_score',ascending=False)
# One pre-specified ensemble: equal averaging of the top 3 distinct configurations by robust score.
top=list(ranking.index[:3]);bp=sum(oof[n] for n in top)/3;bsp=sum(stress_oof[n] for n in top)/3
bc=[f1_score(yi[dev[b]],bp[b].argmax(axis=1),average='macro',zero_division=0) for _,b in folds]
bs=[f1_score(yi[dev[b]],bsp[b].argmax(axis=1),average='macro',zero_division=0) for _,b in folds]
ranking.loc['blend_top3']=[np.mean(bc),np.mean(bs),(np.mean(bc)+np.mean(bs))/2]
ranking=ranking.sort_values('robust_score',ascending=False);ranking.to_csv(OUT/'model_comparison.csv')
selected=top if ranking.index[0]=='blend_top3' else [ranking.index[0]]
print('SELECTED:',selected);display(ranking)
np.savez_compressed(OUT/'oof_probabilities.npz',labels=labels,dev=dev,**{n:oof[n] for n in oof})

canonical_tfidf_lr_c025 0 clean 0.3396 stress 0.2891
canonical_tfidf_lr_c1 0 clean 0.3806 stress 0.3265
canonical_tfidf_lr_c025 1 clean 0.3281 stress 0.2807
canonical_tfidf_lr_c1 1 clean 0.3722 stress 0.3138
canonical_tfidf_lr_c025 2 clean 0.36 stress 0.3053
canonical_tfidf_lr_c1 2 clean 0.3903 stress 0.3359
Initial 10 configurations reused from completed matching group CV; two regularization settings additionally evaluated.
FINAL SELECTED: ['canonical_tfidf_lr_c1']
                         clean_macro_f1  stress_macro_f1  robust_score
model                                                                 
canonical_tfidf_lr_c1          0.381004         0.325403      0.353204
canonical_tfidf_lr             0.385087         0.311318      0.348202
blend_top3                     0.379663         0.308139      0.343901
canonical_tfidf_lr_c16         0.373681         0.296840      0.335261
canonical_tfidf_lr_c64         0.365529         0.292794      0.329162
canonical_lgb15                0

## 4. 이전부터 관측된 홀드아웃의 재평가
선택 이후에만 평가합니다. 이전 실험에서 이미 본 홀드아웃이므로 완전히 새로운 독립 성능으로 해석하지 않습니다. 선택에는 아래 결과를 사용하지 않습니다. 이전 모델의 기록된 clean 홀드아웃 점수와 비교하며, 이전 모델의 스트레스 점수는 추정하지 않습니다.

In [5]:
dev_members=[fit_member(name,dev) for name in selected]
hp=sum(predict_member(m,[rows[m['mode']][i] for i in hold]) for m in dev_members)/len(dev_members)
hpred=labels[hp.argmax(axis=1)]
metrics={'macro_f1':f1_score(y.iloc[hold],hpred,average='macro',zero_division=0),'accuracy':accuracy_score(y.iloc[hold],hpred),'weighted_f1':f1_score(y.iloc[hold],hpred,average='weighted',zero_division=0),'n':len(hold)}
hreport=pd.DataFrame(classification_report(y.iloc[hold],hpred,output_dict=True,zero_division=0)).T
hreport.to_csv(OUT/'holdout_classification_report.csv')
pd.DataFrame({'ID':train.ID.iloc[hold].values,'true':y.iloc[hold].values,'prediction':hpred}).to_csv(OUT/'holdout_predictions.csv',index=False)
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics,indent=2))
print('V2 holdout (previously observed):',metrics);display(hreport)
del dev_members

V2 holdout (previously observed): {'macro_f1': 0.37584306853496, 'accuracy': 0.37711522965350525, 'weighted_f1': 0.3743999207032669, 'n': 1241}
              precision    recall  f1-score      support
ACC            0.916667  0.785714  0.846154    14.000000
BLCA           0.380952  0.400000  0.390244    20.000000
BRCA           0.666667  0.407643  0.505929   157.000000
CESC           0.090909  0.096774  0.093750    31.000000
COAD           0.703704  0.863636  0.775510    44.000000
DLBC           0.800000  0.571429  0.666667     7.000000
GBMLGG         0.647059  0.119565  0.201835    92.000000
HNSC           0.270270  0.227273  0.246914    44.000000
KIPAN          0.314286  0.106796  0.159420   103.000000
KIRC           0.439394  0.432836  0.436090    67.000000
LAML           0.450000  0.290323  0.352941    31.000000
LGG            0.522388  0.760870  0.619469    46.000000
LIHC           0.325000  0.419355  0.366197    31.000000
LUAD           0.136364  0.166667  0.150000    36.000000
L

## 5. 전체 재학습 및 제출
선택한 설정을 전체 train으로 재학습합니다. 최종 후보들은 서로 다른 표현/알고리즘을 결합할 수 있지만 CV 폴드별 모델 평균 대신 전체 데이터 모델을 사용해 이전 full-train 결과와 조건을 맞춥니다. 테스트는 이 단계의 추론에만 사용하고 제출 확률·ID를 검증합니다. 피클은 cloudpickle로 전처리 코드를 포함하여 저장합니다.

In [6]:
members=[fit_member(name,np.arange(len(train))) for name in selected]
def predict_bundle(bundle,frame):
 cache={m['mode']:row_features(frame[bundle['genes']],m['mode']) for m in bundle['members']}
 return sum(predict_member(m,cache[m['mode']]) for m in bundle['members'])/len(bundle['members'])
bundle={'version':2,'members':members,'classes':labels,'genes':genes,'selected':selected,'seed':SEED,'holdout_metrics':metrics,'selection':'mean(clean,synthetic_stress) development group CV','predict':predict_bundle}
model_path=OUT/'cancer_model_v2.pkl'
with model_path.open('wb') as f:cloudpickle.dump(bundle,f,protocol=pickle.HIGHEST_PROTOCOL)
with model_path.open('rb') as f:loaded=pickle.load(f)
tp=predict_bundle(loaded,test)
assert np.isfinite(tp).all() and np.allclose(tp.sum(axis=1),1)
assert np.allclose(tp[:4],predict_bundle(bundle,test.iloc[:4]))
submission=sample.copy();submission['SUBCLASS']=labels[tp.argmax(axis=1)]
assert submission.ID.equals(sample.ID) and submission.SUBCLASS.isin(labels).all()
submission.to_csv(OUT/'submission_v2.csv',index=False)
# Preserve the previously submitted file; publish the new candidate under a distinct name.
submission.to_csv(SUBMIT/'혜진_submission_v2.csv',index=False)
pd.DataFrame(tp,columns=labels,index=test.ID).to_csv(OUT/'test_probabilities.csv')
feature_tables=[]
for m in members:
 est=m['estimator'];tab=pd.DataFrame({'model':m['name'],'feature':m['map'].names})
 if hasattr(est,'feature_importances_'):tab['importance']=est.feature_importances_
 else:tab['importance']=np.abs(est.coef_).mean(axis=0)
 feature_tables.append(tab)
features=pd.concat(feature_tables,ignore_index=True);features.to_csv(OUT/'final_features.csv',index=False)
print('MODEL:',model_path.resolve(),'MiB:',round(model_path.stat().st_size/1024**2,2));display(FileLink(str(model_path)))
print('NEW SUBMISSION:',(SUBMIT/'혜진_submission_v2.csv').resolve())
display(submission.SUBCLASS.value_counts())

MODEL: /Users/mac/project/cancer_project/artifacts/v2/cancer_model_v2.pkl MiB: 16.44
/Users/mac/project/cancer_project/artifacts/v2/cancer_model_v2.pkl
NEW SUBMISSION: /Users/mac/project/cancer_project/submission/혜진_submission_v2.csv
SUBCLASS
COAD      557
STES      231
BRCA      148
OV        146
SKCM      128
TGCT      108
KIPAN     107
UCEC       92
PRAD       92
CESC       87
THYM       86
KIRC       86
LGG        80
HNSC       64
THCA       64
PCPG       63
LUAD       63
SARC       59
LUSC       52
BLCA       49
LIHC       45
GBMLGG     44
PAAD       41
LAML       32
ACC        17
DLBC        5
Name: count, dtype: int64


## 6. 상세 결과 보고서와 독립 로딩 검증
이전 점수와 새 로컬 결과를 구분하고, 실제 새 대회 점수는 제출 이후에만 판단합니다. 노트북과 데이터 외에 프로젝트 .py 파일을 요구하지 않습니다.

In [7]:
def md_table(df):
 return '| '+' | '.join(map(str,df.columns))+' |\n| '+' | '.join(['---']*len(df.columns))+' |\n'+'\n'.join('| '+' | '.join(f'{v:.5f}' if isinstance(v,float) else str(v) for v in r)+' |' for r in df.itertuples(index=False,name=None))
text=f"""# 혜진 암종 분류 v2 개선 보고서

## 결과와 해석
이전 제출 Public macro F1은 **0.2765425197**, 이전 로컬 홀드아웃 macro F1은 **0.4237006861**이었다. 새로운 Public 성능은 아직 측정하지 않았다. 로컬 점수 상승을 대회 점수 상승으로 표현하지 않는다.

선택 모델: {', '.join(selected)}. 개발 CV와 class-independent 배경 노이즈 스트레스의 macro F1 평균을 사용했다. 예전부터 관측된 홀드아웃 결과: {json.dumps(metrics,ensure_ascii=False)}. 이 홀드아웃은 새로운 독립 검증이 아니다. 전체 학습으로 재학습한 피클은 {model_path.name}, 새 제출 파일은 혜진_submission_v2.csv이다. 이전 제출 파일을 덮어쓰지 않았다.

## 확인된 문제와 변경
제출 CSV는 이전 artifacts/submission.csv와 일치했다. ID 정렬 오류는 발견하지 못했다. 데이터의 별도 출처/주석 방식 차이는 원인 가설이며, 대회 정답이 없어 인과관계는 확정할 수 없다.

* 아미노산 위치 뒤 X와 * 종결 표기를 통일했다. R213X와 R213*가 같은 stop 특성을 갖는다.
* K16Rfs/K16fs 같은 프레임시프트 표기를 정규화했다. 이는 전체 전사체/유전체 좌표 정규화가 아니다.
* 유전자별 유형 존재 여부를 사용해 여러 토큰과 반복 표기의 개수에 덜 의존한다. 여러 전사체임을 원본 데이터만으로 확정하지 않는다.
* 원시 토큰 수, 복수 이벤트 수, 평균 위치 등 출처/주석 차이에 민감한 전역 집계를 제거했다.
* gene, functional, canonical 표현, TF-IDF 정규화, LR, LightGBM, ExtraTrees와 학습 내 배경 증강을 비교했다.
* chi2는 폴드 학습 정답으로만 적합하고 상위 2,000개로 제한한다. 검증/테스트 정답은 사용하지 않는다.

## 데이터와 EDA
ID는 샘플 식별자이므로 모델 제외, SUBCLASS는 26종 정답이므로 입력 제외, 유전자 4,384개는 WT/변이 표기다. 결측은 변이 이벤트로 세지 않는다. 변이 유전자 수를 임상 TMB로 부르지 않는다.

{md_table(shift)}

학습/테스트 통계는 EDA와 파서 진단에만 사용했다. 테스트 행을 학습, 단어 사전, 빈도 필터, TF-IDF, chi2, 증강 확률, 모델 선택, 확률 보정에 사용하지 않았다. 증강은 학습 폴드 유전자 출현 확률과 고정 Poisson(20) 배경만 사용했다.

## 파생변수 사전
| 접두사 | 의미 | 근거와 한계 |
| --- | --- | --- |
| gene:G | 유전자 G에 변이 표기 존재 | 표기 상세 차이에 비교적 안정, 변이 종류 소실 |
| type:G:K | G에서 유형 K가 하나 이상 존재 | synonym/substitution/stop/frameshift/deletion/insertion/other의 집계, 기능 판정 아님 |
| variant:G:T | 정규화된 토큰 T 존재 | 반복 hotspot 정보, 미등록 토큰/전사체 차이에 취약 |
| position:G:N | G의 숫자 위치 N 존재 | 같은 위치의 다른 치환 공유, 전사체 불일치 가능 |

빈도 필터는 훈련 환자 3명 이상 출현한 변수만 유지한다. TF-IDF는 훈련 빈도와 L2 행 정규화로 변이량의 영향을 줄인다. 증강을 쓰는 후보도 변수 사전은 실제 훈련 행만으로 먼저 적합한다. 최종 변수 전체는 artifacts/v2/final_features.csv에 기록했다.

## CV 모델 비교
{md_table(ranking.reset_index())}

14개 단일 설정과 상위 3개 동일 가중치 앙상블 1개를 비교했다. blend_top3은 같은 CV 결과를 이용해 구성을 고르므로 선택 편향이 있을 수 있다. 스트레스는 고정된 합성 분포이며 실제 테스트와 같지 않다. 높은 스트레스 점수만으로 대회 점수를 예측하지 않는다.

## 홀드아웃 클래스별 결과
{md_table(hreport.reset_index())}

## 최종 변수
{md_table(features.groupby('model').size().rename('count').reset_index())}

LR importance는 클래스별 계수 절댓값 평균, LightGBM/ExtraTrees는 트리 중요도다. 서로 다른 모델/전처리 간 중요도를 직접 비교하지 않는다. 상관된 변수와 카디널리티의 영향을 받는다. 인과적 중요도가 아니다.

## 실행과 저장
cencer.ipynb에서 Run All. 공용 data/를 상위 폴더에서 찾고 개인 결과는 models/혜진/artifacts/v2, 새 제출은 공용 submission/혜진_submission_v2.csv에 저장한다. 원래 로컬 작업 구조에서는 artifacts/v2를 사용한다. 피클 로딩은 cloudpickle이 설치된 같은 라이브러리 환경에서 pickle.load로 가능하며 bundle['predict'](bundle, dataframe)을 호출한다. 다른 프로젝트 Python 파일은 불필요하다.

## 평가 출처 및 한계
[공식 DACON 규칙](https://dacon.io/competitions/official/236355/overview/rules)에 Macro F1과 Public 30%, Private 100%가 명시되어 있다. 외부 데이터/사전학습/의사 라벨/테스트 적합은 사용하지 않았다. 이전 관측 홀드아웃의 재사용, 제한된 후보 범위, 원본의 상충 중복 라벨, 새 Public 점수 미측정, 합성 스트레스의 현실성 한계가 있다. 어떤 모델의 새 Public 점수도 보장하지 않는다.
"""
(WORK/'improvement_report.md').write_text(text)
print('Report:',WORK/'improvement_report.md')
import subprocess,tempfile
program="""
import pickle,sys,json,numpy as np,pandas as pd
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
d=pd.read_csv(sys.argv[2],nrows=4);p=b['predict'](b,d)
assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
assert not {'mutation_model','report_writer'}.intersection(sys.modules)
print(json.dumps({'status':'passed','rows':len(d),'version':b['version']}))
"""
env=os.environ.copy();env.pop('PYTHONPATH',None)
with tempfile.TemporaryDirectory() as td:
 result=subprocess.run([sys.executable,'-c',program,str(model_path.resolve()),str((REPO/'data/test.csv').resolve())],cwd=td,env=env,capture_output=True,text=True,check=True)
check=json.loads(result.stdout);(OUT/'standalone_check.json').write_text(json.dumps(check,indent=2));print('Standalone pickle check:',check)
metadata={'python':sys.version,'holdout':metrics,'selected':selected,'model_bytes':model_path.stat().st_size,'submission_rows':len(submission),'independent_pickle_check':check}
(OUT/'verification.json').write_text(json.dumps(metadata,indent=2))

Report: /Users/mac/project/cancer_project/improvement_report.md
Standalone pickle check: {'status': 'passed', 'rows': 4, 'version': 2}
